In [48]:
import torch
import torch.optim as optim
from transformers import BartTokenizer, BartForConditionalGeneration
import time
from typing import List, Tuple
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Load pre-trained BART model and tokenizer
model_name = "facebook/bart-base"  # or "facebook/bart-large" for more capacity
print(f"Loading model: {model_name}...")
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)

print(f"Model loaded: {model_name}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")
print()

Using device: cuda
GPU: NVIDIA GeForce RTX 3080
Memory allocated: 0.57 GB
Loading model: facebook/bart-base...
Model loaded: facebook/bart-base
Model parameters: 139,420,416
Tokenizer vocab size: 50,265



In [16]:
generate_response("whoa re you")

'whoa re you'

In [ ]:
# 1. Load your data
df = pd.read_csv("../data/processed/mr_all.csv")  # Should have 'message' and 'response' columns
print(f"Loaded {len(df)} message-response pairs")

# 2. Create Dataset class
class ConversationDataset(Dataset):
    def __init__(self, messages, responses):
        self.messages = messages
        self.responses = responses
    
    def __len__(self):
        return len(self.messages)
    
    def __getitem__(self, idx):
        return self.messages[idx], self.responses[idx]

# 3. Setup device, tokenizer, model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base").to(device)

# 4. Create DataLoader with collate function
def collate_fn(batch):
    """Batch processing function"""
    messages, responses = zip(*batch)
    
    # Tokenize messages (inputs)
    inputs = tokenizer(
        list(messages), 
        padding=True, 
        truncation=True, 
        max_length=64, 
        return_tensors="pt"
    )
    
    # Tokenize responses (targets)
    targets = tokenizer(
        list(responses), 
        padding=True, 
        truncation=True, 
        max_length=64, 
        return_tensors="pt"
    )
    
    # Replace padding tokens with -100 (ignored in loss)
    labels = targets["input_ids"].clone()
    labels[labels == tokenizer.pad_token_id] = -100
    
    return {
        "input_ids": inputs["input_ids"].to(device),
        "attention_mask": inputs["attention_mask"].to(device),
        "labels": labels.to(device)
    }

# 5. Create Dataset and DataLoader
dataset = ConversationDataset(
    messages=df['message'].tolist(),
    responses=df['response'].tolist()
)

dataloader = DataLoader(
    dataset, 
    batch_size=8,  # Adjust based on your GPU memory
    shuffle=True, 
    collate_fn=collate_fn
)

# 6. Training setup
optimizer = optim.AdamW(model.parameters(), lr=3e-5)
model.train()

print("\nStarting training...")

# 7. Training loop
for epoch in range(5):  # 5 epochs
    total_loss = 0
    for batch_idx, batch in enumerate(dataloader):
        # Forward pass
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        
        # Backward pass
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
        
        # Print progress every 10 batches
        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}")
    
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}")
    
    # 8. Test after each epoch
    model.eval()
    test_messages = [
        "What time are we meeting tomorrow?",
        "Hey, how's it going?",
        "Did you finish the project?"
    ]
    
    print("\nTesting after epoch", epoch + 1)
    for msg in test_messages:
        inputs = tokenizer(msg, return_tensors="pt", truncation=True, max_length=64).to(device)
        
        with torch.no_grad():
            output_ids = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=50,
                num_beams=3,
                early_stopping=True
            )
        
        response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        print(f"  Input: {msg}")
        print(f"  Response: {response}")
    print()
    
    model.train()

# 9. Save the model
model.save_pretrained("./models/bart_finetuned")
tokenizer.save_pretrained("./models/bart_finetuned")
print("Model saved to ./models/bart_finetuned/")

# 10. Test function
def generate_response(message):
    model.eval()
    inputs = tokenizer(message, return_tensors="pt", truncation=True, max_length=64).to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=50,
            num_beams=3,
            early_stopping=True,
            no_repeat_ngram_size=2  # Avoid repetition
        )
    
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 11. Final test
print("\n" + "="*50)
print("FINAL TEST")
print("="*50)

test_messages = [
    "What time are we meeting tomorrow?",
    "Hey, how's it going?",
    "Can you send me the document?",
    "I'll be there in 10 minutes"
]

for msg in test_messages:
    response = generate_response(msg)
    print(f"Input: {msg}")
    print(f"Response: {response}")
    print("-" * 40)

Loaded 4620 message-response pairs
Using device: cuda

Starting training...
Epoch 1, Batch 0, Loss: 8.2288
Epoch 1, Batch 10, Loss: 6.2084
Epoch 1, Batch 20, Loss: 5.5254
Epoch 1, Batch 30, Loss: 4.9684
Epoch 1, Batch 40, Loss: 5.2451
Epoch 1, Batch 50, Loss: 5.4492
Epoch 1, Batch 60, Loss: 4.9065
Epoch 1, Batch 70, Loss: 4.6121
Epoch 1, Batch 80, Loss: 4.9319
Epoch 1, Batch 90, Loss: 5.1862
Epoch 1, Batch 100, Loss: 4.6664
Epoch 1, Batch 110, Loss: 5.2931
Epoch 1, Batch 120, Loss: 4.3829
Epoch 1, Batch 130, Loss: 4.7741
Epoch 1, Batch 140, Loss: 4.5742
Epoch 1, Batch 150, Loss: 4.9614
Epoch 1, Batch 160, Loss: 3.6916
Epoch 1, Batch 170, Loss: 4.4938
Epoch 1, Batch 180, Loss: 4.2644
Epoch 1, Batch 190, Loss: 3.8697
Epoch 1, Batch 200, Loss: 4.3697
Epoch 1, Batch 210, Loss: 4.8765
Epoch 1, Batch 220, Loss: 4.6552
Epoch 1, Batch 230, Loss: 3.7212
Epoch 1, Batch 240, Loss: 4.6907
Epoch 1, Batch 250, Loss: 4.0664
Epoch 1, Batch 260, Loss: 4.0642
Epoch 1, Batch 270, Loss: 4.3654
Epoch 1, Ba

c:\Users\fhjrj\miniconda3\envs\ml_gpu\lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Model saved to ./bart_finetuned/

FINAL TEST
Input: What time are we meeting tomorrow?
Response: what day is it
----------------------------------------
Input: Hey, how's it going?
Response: yeah it is
----------------------------------------
Input: Can you send me the document?
Response: send me the document bro
----------------------------------------
Input: I'll be there in 10 minutes
Response: i'll be there in 10 minutes
----------------------------------------


In [52]:
# Add this at the end of your training script

# Switch to evaluation mode
model.eval()

# Test with sample messages
test_messages = [
    "What time are we meeting tomorrow?",
    "Hey, how's it going?",
    "Did you finish the project?",
    "🧌 🧌 🧌 🧌 🧌 🧌 🧌 🧌 🧌 🧌 🧌",
    "yo",
    "you gaming or not?",
    "you can buy them outside",
    "please buy it yourself",
    "you gaming or not"
]

for msg in test_messages:
    inputs = tokenizer(msg, return_tensors="pt", truncation=True, max_length=64).to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=50,
            num_beams=3,
            early_stopping=True
        )
    
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"Input: {msg}")
    print(f"Response: {response}\n")

Input: What time are we meeting tomorrow?
Response: what day is it

Input: Hey, how's it going?
Response: yeah it is

Input: Did you finish the project?
Response: did you finish the project?

Input: 🧌 🧌 🧌 🧌 🧌 🧌 🧌 🧌 🧌 🧌 🧌
Response: what the hael

Input: yo
Response: what the hael

Input: you gaming or not?
Response: nah bro i'm gaming

Input: you can buy them outside
Response: i don't think so

Input: please buy it yourself
Response: i'm not buying it

Input: you gaming or not
Response: gaming or not

